# CMIP6 MMM is resolution-invariant: 0.25° ≡ 1°

**Claim.** The CMIP6 multi-model mean (MMM), regridded to 0.25°, carries **no information beyond**
the same MMM at 1°. CMIP6 models are natively coarse (~0.7–2.8°), so interpolating to a finer target
grid cannot invent sub-grid structure — the 0.25° field is just a smooth up-sampling of the 1° one.

**Proof strategy.** Build the MMM two ways:
1. regrid each model to **0.25°**, average → `MMM_025`;
2. regrid each model to **1°**, average → `MMM_1`.

Then block-average `MMM_025` back to 1° (`MMM_025→1`) and compare it with `MMM_1`. If the claim
holds, the difference is at the level of interpolation round-off — orders of magnitude below the
field's own spatial variability — and the two are ~perfectly correlated.

This is why the sensitivity notebooks render the CMIP6 benchmark at 1° only.

> **Where to run.** Loads every configured CMIP6 model — DKRZ **compute node**, `feather` env.

In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import nereus as nr

from feather.config import FeatherConfig
from feather.data.cmip6 import CMIP6Loader

plt.rcParams["figure.dpi"] = 110

## 1. Configuration & loader

`VAR` is the CMIP6 variable (default `tas`; try `pr` too). We use the same influence-radius floor
(250 km) the framework uses for CMIP6 regridding.

In [ ]:
CONFIG_PATH = "../configs/eerie.yaml"
VAR, TABLE = "tas", "Amon"
RES_HI, RES_LO = 0.25, 1.0
COARSEN = int(round(RES_LO / RES_HI))

cfg = FeatherConfig.from_yaml(CONFIG_PATH)
PERIOD = cfg.get_period()
INFL = max(float(cfg.nereus.get("influence_radius", 80_000.0)), 250_000.0)

cmip6_loader = CMIP6Loader(cfg)
print("period :", PERIOD)
print("CMIP6  :", list(cmip6_loader.models))
print("influence radius (m):", INFL)

## 2. Regrid helper

In [ ]:
def regrid_field(field, lon, lat, resolution, influence_radius=INFL, method="linear"):
    vals = np.asarray(field).ravel()
    lon = np.asarray(lon)
    lat = np.asarray(lat)
    if lon.ndim == 1 and lon.shape[0] != vals.shape[0]:
        lon, lat = np.meshgrid(lon, lat)
    _, interp = nr.regrid(
        vals, lon=np.asarray(lon), lat=np.asarray(lat),
        resolution=resolution, method=method,
        influence_radius=influence_radius,
        lon_bounds=(0.0, 360.0), as_xarray=True,
    )
    tlat = np.asarray(interp.target_lat)[:, 0]
    tlon = np.asarray(interp.target_lon)[0, :]
    return xr.DataArray(interp(vals), dims=("lat", "lon"),
                        coords={"lat": tlat, "lon": tlon}, name=VAR)

## 3. Build the MMM at both resolutions

Regrid every model to 0.25° and to 1° (same climatology, two targets), then average across models.

In [ ]:
def latlon(da):
    la = da["lat"].values if "lat" in da.coords else da["latitude"].values
    lo = da["lon"].values if "lon" in da.coords else da["longitude"].values
    return lo, la

mem_hi, mem_lo, used = [], [], []
for cm in cmip6_loader.models:
    da = cmip6_loader.load_var(VAR, cm, table=TABLE, period=PERIOD, time_mean=True)
    if da is None:
        print(f"  skip {cm}: no data")
        continue
    lo, la = latlon(da)
    try:
        rhi = regrid_field(da.values, lo, la, RES_HI)
        rlo = regrid_field(da.values, lo, la, RES_LO)
    except Exception as e:  # noqa: BLE001
        print(f"  skip {cm}: {e}")
        continue
    mem_hi.append(rhi)
    mem_lo.append(rlo)
    used.append(cm)

mmm_hi = xr.concat(mem_hi, dim="model").mean("model")   # 0.25° MMM
mmm_lo = xr.concat(mem_lo, dim="model").mean("model")   # 1° MMM
print(f"MMM from {len(used)} models | 0.25° {mmm_hi.shape} | 1° {mmm_lo.shape}")

## 4. Bring 0.25° MMM to the 1° grid and compare

Block-average `MMM_025` to 1° and diff against `MMM_1`. We report max/mean absolute difference,
RMS difference, its ratio to the field's spatial std, and the Pearson correlation.

In [ ]:
mmm_hi_on_lo = (mmm_hi.coarsen(lat=COARSEN, lon=COARSEN, boundary="trim").mean()
                .interp(lat=mmm_lo.lat, lon=mmm_lo.lon))
diff = mmm_hi_on_lo - mmm_lo

a = mmm_hi_on_lo.values.ravel()
b = mmm_lo.values.ravel()
ok = np.isfinite(a) & np.isfinite(b)
a, b = a[ok], b[ok]
dvals = a - b

field_std = float(np.std(b))
rms = float(np.sqrt(np.mean(dvals ** 2)))
corr = float(np.corrcoef(a, b)[0, 1])

print(f"variable            : {VAR}")
print(f"field spatial std   : {field_std:.4g}")
print(f"max |diff|          : {np.abs(dvals).max():.4g}")
print(f"mean |diff|         : {np.abs(dvals).mean():.4g}")
print(f"RMS diff            : {rms:.4g}")
print(f"RMS diff / std      : {rms / field_std * 100:.3f} %")
print(f"Pearson correlation : {corr:.8f}")

## 5. Maps: the two MMMs and their (tiny) difference

Left/middle share a colour scale; the right panel is on a scale ~100× tighter — note the difference
is essentially interpolation texture, not physical structure.

In [ ]:
vmin = float(np.nanpercentile(mmm_lo, 2))
vmax = float(np.nanpercentile(mmm_lo, 98))
dmax = float(np.nanpercentile(np.abs(diff), 99)) or 1e-6

panels = [("MMM 1°", mmm_lo, "viridis", vmin, vmax),
          ("MMM 0.25° → 1°", mmm_hi_on_lo, "viridis", vmin, vmax),
          ("difference (0.25°→1° − 1°)", diff, "RdBu_r", -dmax, dmax)]
fig, axes = plt.subplots(1, 3, figsize=(18, 4.2),
                         subplot_kw={"projection": ccrs.Robinson()})
for ax, (ttl, fld, cmap, lo, hi) in zip(axes, panels):
    p = ax.pcolormesh(fld.lon, fld.lat, fld, cmap=cmap, vmin=lo, vmax=hi,
                      shading="auto", transform=ccrs.PlateCarree())
    ax.coastlines(linewidth=0.4)
    ax.set_title(ttl, fontsize=10)
    fig.colorbar(p, ax=ax, orientation="horizontal", pad=0.04, shrink=0.85)
fig.suptitle(f"CMIP6 MMM {VAR}: 0.25° vs 1° are the same field", y=1.03)
plt.show()

## 6. Scatter & histogram of the difference

A 1:1 scatter (should sit exactly on the diagonal) and a histogram of the pointwise difference
(should be a sharp spike at zero).

In [ ]:
fig, (axs, axh) = plt.subplots(1, 2, figsize=(13, 5))

axs.scatter(b, a, s=2, alpha=0.2, color="#1f77b4")
lim = [min(a.min(), b.min()), max(a.max(), b.max())]
axs.plot(lim, lim, "k--", lw=1)
axs.set_xlabel(f"MMM 1°  [{VAR}]")
axs.set_ylabel(f"MMM 0.25° → 1°  [{VAR}]")
axs.set_title(f"1:1 agreement (r = {corr:.6f})")

axh.hist(dvals, bins=120, color="#8da0cb")
axh.set_xlabel(f"difference  [{VAR}]")
axh.set_ylabel("1° grid cells")
axh.set_title(f"RMS = {rms:.3g}  ({rms / field_std * 100:.3f}% of field std)")
axh.set_yscale("log")
fig.tight_layout()
plt.show()

## 7. Conclusion

`MMM_025→1` and `MMM_1` agree to within a small fraction of a percent of the field's own spatial
variability, with correlation ≈ 1. The residual is pure regridding round-off (linear interpolation
to 0.25° then block-averaging back is not algebraically identical to interpolating straight to 1°),
**not** physical fine-scale signal — CMIP6 has none to give at those scales.

**Implication.** Rendering the CMIP6 benchmark at 0.25° is wasted computation and can even look
misleadingly smooth-but-detailed; the 1° MMM is the honest, cheaper representation. Any genuine
sub-1° structure in a resolution comparison must come from the high-resolution models, not from
regridding the coarse ensemble.